# 🌊 Day 5 — Pipeline Automation & Data Quality
## Caspian Maritime Delay-Risk Forecasting

**Capstone of Week 1.** Stitch ingestion → cleaning → features → training → prediction into one runnable pipeline with automated quality gates.

---

### 📌 Today's Agenda

| # | Task | Deliverable |
|---|------|-------------|
| 1 | Pipeline Orchestrator | `src/pipeline.py` with `--mode full` / `--mode incremental` |
| 2 | Incremental Loading | `INSERT OR REPLACE` + per-city max-date detection |
| 3 | Automated Quality Gates | `src/quality_checks.py` with 6 checks |
| 4 | Logging | Rotating file handler at `logs/pipeline.log` |
| 5 | End-to-end run + architecture diagram | This notebook |

### ⏰ Automation

Scheduled automatically via **GitHub Actions** (`.github/workflows/monthly-pipeline.yml`) — runs on the 1st of every month at 06:00 UTC on GitHub's servers. No local PC needed.

---
## 0 — Environment Setup

In [1]:
import sys, logging, warnings
from pathlib import Path
from datetime import datetime

import pandas as pd

warnings.filterwarnings('ignore')

REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Force clean imports — notebooks often cache old versions
for m in list(sys.modules):
    if m.startswith('src'):
        del sys.modules[m]

from src.pipeline import run_pipeline
from src.quality_checks import run_all_checks, format_check_report
from src.database import get_connection, run_query
from src.config import PATHS

DB_PATH = PATHS['repo_root'] / 'data' / 'caspian_weather.duckdb'
print(f'✅ Ready. DB: {DB_PATH}')

✅ Ready. DB: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\data\caspian_weather.duckdb


---
## 1 — Pipeline Architecture

```
                            ┌──────────────────┐
                            │  GitHub Actions  │
                            │  cron: 0 6 1 * * │
                            └────────┬─────────┘
                                     │ triggers monthly
                                     ▼
             ┌───────────────────────────────────────────┐
             │          src/pipeline.py  main()          │
             │     --mode {full | incremental}            │
             └───────────────────────┬───────────────────┘
                                     ▼
   ┌─────────────────────────────────────────────────────────────┐
   │                                                             │
   │  ┌─────────┐   ┌──────────┐   ┌─────────┐   ┌────────────┐  │
   │  │ INGEST  │─▶│ LOAD RAW │─▶│  CLEAN  │─▶│  FEATURES  │  │
   │  │ Open-   │   │  CSV →   │   │  raw →  │   │ staging →  │  │
   │  │ Meteo   │   │  DuckDB  │   │ staging │   │ analytics  │  │
   │  └────┬────┘   └────┬─────┘   └────┬────┘   └─────┬──────┘  │
   │       │             │              │              │         │
   │     ❬gate❭       ❬gate❭         ❬gate❭         ❬gate❭       │
   │       │             │              │              │         │
   │       ▼             ▼              ▼              ▼         │
   │   row_count    freshness       null_ratio    feature_       │
   │                                continuity    completeness   │
   │                                value_ranges                 │
   │                                                             │
   │  ┌─────────┐   ┌──────────────┐                             │
   │  │ TRAIN   │─▶│  PREDICT     │                             │
   │  │ model   │   │  next month  │                             │
   │  └─────────┘   └──────┬───────┘                             │
   │                       │                                     │
   │                       ▼                                     │
   │              predictions/YYYY-MM.csv                        │
   │                                                             │
   │  Run metadata → meta.pipeline_runs                          │
   │  Logs         → logs/pipeline.log                           │
   │  Quality flags → meta.quality_flags                         │
   │                                                             │
   └─────────────────────────────────────────────────────────────┘
```

### Stage ⇄ Gate Mapping

| Stage | Gate | Severity | Trigger |
|-------|------|----------|---------|
| Raw load | `row_count` | **ABORT** | 0 rows loaded |
| Raw load | `freshness_monthly` | WARN | Latest date > 35 days old |
| Staging  | `null_ratio` | WARN | Any column > 5% nulls |
| Staging  | `date_continuity` | WARN | Any gap > 3 days |
| Staging  | `value_ranges` | FLAG | Out-of-range rows → `meta.quality_flags` |
| Analytics | `feature_completeness` | WARN | Required features missing/null |

### Incremental Window Logic

```
  incremental mode:  window = [min(max_date_per_city) - 3 days, yesterday]
  full mode:         window = [DATE_RANGE.start, yesterday]
  manual:            window = [--since, yesterday]
```

The 3-day overlap is intentional — it lets `INSERT OR REPLACE` self-heal bad data from the previous run.

---
## 2 — Full Pipeline Run

Run in `full` mode. Ingests from `DATE_RANGE.start` (2015-01-01), builds the entire database, trains the (stub) model, generates predictions.

In [2]:
print('╔' + '═' * 68 + '╗')
print('║  Run #1 — FULL MODE' + ' ' * 48 + '║')
print('╚' + '═' * 68 + '╝\n')

summary_full = run_pipeline(mode='full', fetch=False, skip_predict=False, skip_train=False)

print('\n━' * 35)
print('RUN SUMMARY')
print('─' * 70)
for k, v in summary_full.items():
    if k == 'check_results':
        continue
    print(f'  {k:<20}: {v}')

╔════════════════════════════════════════════════════════════════════╗
║  Run #1 — FULL MODE                                                ║
╚════════════════════════════════════════════════════════════════════╝

2026-05-03 19:14:53  [20260503_191453]  INFO     pipeline: ╔════════════════════════════════════════════════════════════════════╗
2026-05-03 19:14:53  [20260503_191453]  INFO     pipeline: ║  Pipeline run: 20260503_191453  (mode=full, dry_run=False)
2026-05-03 19:14:53  [20260503_191453]  INFO     pipeline: ╚════════════════════════════════════════════════════════════════════╝
2026-05-03 19:14:53  [20260503_191453]  INFO     src.database: Connected to DuckDB: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\data\caspian_weather.duckdb
2026-05-03 19:14:53  [20260503_191453]  INFO     pipeline: Fetch window: 2015-01-01 → 2026-05-02
2026-05-03 19:14:53  [20260503_191453]  INFO     pipeline: === STAGE 1: Ingest skipped ===
2026

In [3]:
# Inspect the quality check results
print(format_check_report(summary_full['check_results']))

STATUS   SEVERITY STAGE        CHECK                      MESSAGE
─────────────────────────────────────────────────────────────────
✅        ABORT    raw          row_count                  raw.weather_daily: 20,700 rows
✅        WARN     raw          freshness_monthly          Data is 1 days old — within threshold
✅        WARN     staging      null_ratio                 All columns within 5% null threshold
✅        WARN     staging      date_continuity            No gaps > 3 days
✅        FLAG     staging      value_ranges               All values within physical bounds
✅        WARN     analytics    feature_completeness       All 21 required analytics columns present and non-null
✅        WARN     predict      predictions_completeness   All predictions cover the rolling window cleanly


---
## 3 — Incremental Run

Now run in `incremental` mode. Since we just finished a full run, the pipeline should detect there's no new data and exit quickly.

In [4]:
print('╔' + '═' * 68 + '╗')
print('║  Run #2 — INCREMENTAL MODE (should detect nothing new)' + ' ' * 13 + '║')
print('╚' + '═' * 68 + '╝\n')

summary_inc = run_pipeline(mode='incremental')

print('\n━' * 35)
print(f'Status: {summary_inc["status"]}')
print(f'Duration: {summary_inc["duration_sec"]}s')
if summary_inc['status'] == 'UP_TO_DATE':
    print('✅ Pipeline correctly detected no new data and skipped all downstream stages.')


╔════════════════════════════════════════════════════════════════════╗
║  Run #2 — INCREMENTAL MODE (should detect nothing new)             ║
╚════════════════════════════════════════════════════════════════════╝

2026-05-03 19:15:02  [20260503_191502]  INFO     pipeline: ╔════════════════════════════════════════════════════════════════════╗
2026-05-03 19:15:02  [20260503_191502]  INFO     pipeline: ║  Pipeline run: 20260503_191502  (mode=incremental, dry_run=False)
2026-05-03 19:15:02  [20260503_191502]  INFO     pipeline: ╚════════════════════════════════════════════════════════════════════╝
2026-05-03 19:15:03  [20260503_191502]  INFO     src.database: Connected to DuckDB: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\data\caspian_weather.duckdb
2026-05-03 19:15:03  [20260503_191502]  INFO     pipeline: Fetch window: 2026-04-29 → 2026-05-02
2026-05-03 19:15:03  [20260503_191502]  INFO     pipeline: Current max dates: {'Baku': '

In [5]:
import json
import pandas as pd
from src.config import PATHS

preds_dir = PATHS["repo_root"] / "predictions"

latest_path = preds_dir / "latest.json"

if not latest_path.exists():
    raise FileNotFoundError(
        f"{latest_path} does not exist. Run the prediction stage first."
    )

latest = json.loads(latest_path.read_text(encoding="utf-8"))

daily_path = preds_dir / latest["daily_path"]
monthly_path = preds_dir / latest["monthly_path"]

print("latest.json:")
print(json.dumps(latest, indent=2))

print("\nDaily CSV path:", daily_path)
print("Monthly CSV path:", monthly_path)

daily = pd.read_csv(daily_path)
monthly = pd.read_csv(monthly_path)

print("\nSource counts:")
print(daily["source"].value_counts())

print("\nDate range by source:")
print(daily.groupby("source")["date"].agg(["min", "max", "count"]))

print("\nWindow summary:")
display(monthly)

latest.json:
{
  "window_start": "2026-05-03",
  "window_end": "2026-06-02",
  "days": 31,
  "daily_path": "latest/daily.csv",
  "monthly_path": "latest/monthly.csv",
  "generated_at": "2026-05-03T19:15:10"
}

Daily CSV path: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\predictions\latest\daily.csv
Monthly CSV path: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\predictions\latest\monthly.csv

Source counts:
source
short_horizon    80
climatology      75
Name: count, dtype: int64

Date range by source:
                      min         max  count
source                                      
climatology    2026-05-19  2026-06-02     75
short_horizon  2026-05-03  2026-05-18     80

Window summary:


,city,window_start,window_end,risk_days_predicted,high_risk_window_probability,n_short_horizon_days,n_climatology_days
0,Aktau,2026-05-03,2026-06-02,4,0.0792,16,15
1,Anzali,2026-05-03,2026-06-02,2,0.0220,16,15
2,Baku,2026-05-03,2026-06-02,13,0.1946,16,15
3,Makhachkala,2026-05-03,2026-06-02,1,0.0258,16,15
4,Turkmenbashi,2026-05-03,2026-06-02,0,0.0184,16,15


In [6]:
# -------------------------------------------------------------------
# Current production model summary
# -------------------------------------------------------------------
# This reads the actual production artifact: models/daily_model.pkl.
# The comparison report is produced by the SAME function that saves the model:
# src.modeling.train_model().

import pickle
import pandas as pd
from pathlib import Path

model_path = PATHS["models"] / "daily_model.pkl"
comparison_path = PATHS["repo_root"] / "reports" / "day08_model_comparison.csv"

print("=== Current Production Model ===")
model_bundle = {}

if model_path.exists():
    with model_path.open("rb") as f:
        model_bundle = pickle.load(f)

    print(f"Model file:        {model_path}")
    print(f"Model name:        {model_bundle.get('model_name', 'unknown')}")
    print(f"Base model:        {model_bundle.get('base_model_name', 'unknown')}")
    print(f"Calibrated:        {model_bundle.get('is_calibrated', 'unknown')}")
    print(f"Calibration:       {model_bundle.get('calibration_method', 'none')}")
    print(f"Target:            {model_bundle.get('target', 'unknown')}")
    print(f"Decision threshold:{model_bundle.get('decision_threshold', 0.5)}")
    print(f"Trained rows:      {model_bundle.get('trained_rows', 'unknown')}")
    print(f"Positive rate:     {model_bundle.get('positive_rate', 'unknown')}")
    print(f"Feature count:     {len(model_bundle.get('feature_cols', []))}")
    print(f"Report path:       {model_bundle.get('comparison_report_path', comparison_path)}")
else:
    print(f"No saved model found at: {model_path}")

print("\n=== Latest Model Evaluation Report ===")

if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    saved_model_name = model_bundle.get("model_name", None) if model_path.exists() else None

    print(f"Evaluation file:   {comparison_path}")

    row = None
    if saved_model_name and "Model" in comparison.columns:
        matches = comparison[comparison["Model"].astype(str) == str(saved_model_name)]
        if matches.empty:
            norm_saved = str(saved_model_name).lower().replace(" ", "").replace("-", "_")
            norm_col = comparison["Model"].astype(str).str.lower().str.replace(" ", "", regex=False).str.replace("-", "_", regex=False)
            matches = comparison[norm_col == norm_saved]
        if not matches.empty:
            row = matches.iloc[0]

    if row is not None:
        print(f"Reported model:    {row['Model']}")
        if 'Selected' in row:
            print(f"Selected in report:{row['Selected']}")
        if 'Threshold' in row:
            print(f"Threshold:         {row['Threshold']}")
        print(f"Accuracy:          {row['Accuracy']}")
        if 'Precision' in row:
            print(f"Precision:         {row['Precision']}")
        if 'Recall' in row:
            print(f"Recall:            {row['Recall']}")
        print(f"F1:                {row['F1']}")
        print(f"ROC-AUC:           {row['ROC-AUC']}")
        print(f"Brier:             {row['Brier']}")
        if 'Production score' in row:
            print(f"Production score:  {row['Production score']}")
    else:
        print("Saved model was not found in comparison table.")
        print("This means Day 8/train_model has not been rerun after the latest model file change.")
        print("Showing full comparison table instead:")
        display(comparison)
else:
    print(f"No comparison report found at: {comparison_path}")


=== Current Production Model ===
Model file:        C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\models\daily_model.pkl
Model name:        LogisticRegression_calibrated
Base model:        LogisticRegression
Calibrated:        True
Calibration:       sigmoid
Target:            target_risk_next_day
Decision threshold:0.11
Trained rows:      20695
Positive rate:     0.07929451558347426
Feature count:     20
Report path:       C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\reports\day08_model_comparison.csv

=== Latest Model Evaluation Report ===
Evaluation file:   C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\reports\day08_model_comparison.csv
Reported model:    LogisticRegression_calibrated
Selected in report:True
Threshold:         0.11
Accuracy:          0.779
Precision:         0.219
Recall:            0.622
F1:        

In [7]:
import pickle
from src.config import PATHS

model_path = PATHS["models"] / "daily_model.pkl"

with model_path.open("rb") as f:
    bundle = pickle.load(f)

print("model_name:", bundle.get("model_name", type(bundle)))
print("target:", bundle.get("target", "NO TARGET FIELD"))
print("feature count:", len(bundle.get("feature_cols", [])))

print("\nFeatures in saved production model:")
for i, c in enumerate(bundle.get("feature_cols", []), 1):
    print(f"{i:02d}. {c}")

model_name: LogisticRegression_calibrated
target: target_risk_next_day
feature count: 20

Features in saved production model:
01. apparent_temperature_mean
02. low_visibility_recent
03. month_sin
04. precip_3d_sum
05. precip_change_1d
06. relative_humidity_2m_mean
07. shortwave_radiation_sum
08. strong_wind_recent
09. surface_pressure_mean
10. temp_range_c
11. wind_3d_max
12. wind_change_1d
13. wind_precip_combo
14. wave_height_lag1
15. wave_height_lag2
16. wave_height_7d_mean
17. wave_height_7d_max
18. wave_height_30d_mean
19. wave_height_30d_max
20. wave_height_anom


---
## 4 — Inspect Outputs

In [8]:
conn = get_connection(DB_PATH)

# Run history
print('=== meta.pipeline_runs (most recent 5) ===')
print(run_query(conn, '''
    SELECT run_id, mode, status, duration_sec,
           rows_ingested, rows_analytics,
           predictions_for, warnings_count
    FROM meta.pipeline_runs
    ORDER BY run_id DESC
    LIMIT 5
''').to_string(index=False))

2026-05-03 19:15:10  [20260503_191502]  INFO     src.database: Connected to DuckDB: C:\Users\Baku\Desktop\notebooks\Weather Project\Anemoi Weather Project\Maritime-delay-risk-prediction\data\caspian_weather.duckdb
=== meta.pipeline_runs (most recent 5) ===
         run_id        mode  status  duration_sec  rows_ingested  rows_analytics       predictions_for  warnings_count
20260503_191502 incremental SUCCESS           7.3              0           20700 2026-05-03_2026-06-02               0
20260503_191453        full SUCCESS           9.6              0           20700 2026-05-03_2026-06-02               0
20260503_184629 incremental  FAILED           1.8              0           20700                                     0
20260503_184404        full  FAILED         145.2          20700           20700                                     0
20260503_184032 incremental  FAILED           1.8              0           20700                                     0


In [9]:
# Quality flags recorded during the run
print('=== meta.quality_flags (sample) ===')
try:
    flags = run_query(conn, '''
        SELECT column_name, reason, COUNT(*) AS n
        FROM meta.quality_flags
        GROUP BY column_name, reason
        ORDER BY n DESC
        LIMIT 10
    ''')
    if len(flags) > 0:
        print(flags.to_string(index=False))
    else:
        print('✅ No out-of-range values found.')
except Exception as e:
    print(f'ℹ️  No quality flags table (expected if no violations): {e}')

=== meta.quality_flags (sample) ===
✅ No out-of-range values found.


In [10]:
# Latest predictions
preds_dir = PATHS['repo_root'] / 'predictions'
if preds_dir.exists():
    csvs = sorted(preds_dir.glob('*.csv'))
    if csvs:
        latest = csvs[-1]
        print(f'=== Latest predictions: {latest.name} ===')
        print(pd.read_csv(latest).to_string(index=False))
    else:
        print('No prediction CSVs yet.')
else:
    print('predictions/ directory not created yet.')

No prediction CSVs yet.


In [11]:
# Pipeline log (last run's lines)
log_file = PATHS['repo_root'] / 'logs' / 'pipeline.log'
if log_file.exists():
    text = log_file.read_text(encoding='utf-8')
    lines = text.strip().split('\n')
    # Show the last 25 lines
    print(f'=== logs/pipeline.log (last 25 of {len(lines)} lines) ===')
    for line in lines[-25:]:
        print(line)
else:
    print('No log file yet.')

=== logs/pipeline.log (last 25 of 4248 lines) ===
2026-05-03 19:15:07  [20260503_191502]  INFO     src.era5_client:   ✓ 7 forecast days received
2026-05-03 19:15:07  [20260503_191502]  INFO     src.modeling:   Offshore marine forecast fetched for Aktau (7 days)
2026-05-03 19:15:07  [20260503_191502]  INFO     src.era5_client: Fetching marine forecast (37.55, 49.55)  7 days
2026-05-03 19:15:07  [20260503_191502]  INFO     src.era5_client:   ✓ 7 forecast days received
2026-05-03 19:15:07  [20260503_191502]  INFO     src.modeling:   Offshore marine forecast fetched for Anzali (7 days)
2026-05-03 19:15:07  [20260503_191502]  INFO     src.era5_client: Fetching marine forecast (40.05, 53.10)  7 days
2026-05-03 19:15:07  [20260503_191502]  INFO     src.era5_client:   ✓ 7 forecast days received
2026-05-03 19:15:07  [20260503_191502]  INFO     src.modeling:   Offshore marine forecast fetched for Turkmenbashi (7 days)
2026-05-03 19:15:07  [20260503_191502]  INFO     src.era5_client: Fetching mar

---
## 5 — Manual Quality Check Demo

You can also invoke individual checks outside the pipeline (e.g., as part of a notebook-based exploration).

In [12]:
# Run all staging checks
staging_results = run_all_checks(conn, stage='staging')
print(format_check_report(staging_results))

STATUS   SEVERITY STAGE        CHECK                      MESSAGE
─────────────────────────────────────────────────────────────────
✅        WARN     staging      null_ratio                 All columns within 5% null threshold
✅        WARN     staging      date_continuity            No gaps > 3 days
✅        FLAG     staging      value_ranges               All values within physical bounds


In [13]:
from pathlib import Path
from src.config import PATHS

preds = PATHS["repo_root"] / "predictions"

print("latest.json exists:", (preds / "latest.json").exists())
print("latest/daily.csv exists:", (preds / "latest" / "daily.csv").exists())
print("latest/monthly.csv exists:", (preds / "latest" / "monthly.csv").exists())

print("\nPrediction folder contents:")
for p in preds.iterdir():
    print("-", p.name)

latest.json exists: True
latest/daily.csv exists: True
latest/monthly.csv exists: True

Prediction folder contents:
- latest
- latest.json


In [14]:
# Run all analytics checks
analytics_results = run_all_checks(conn, stage='analytics')
print(format_check_report(analytics_results))

conn.close()

STATUS   SEVERITY STAGE        CHECK                      MESSAGE
─────────────────────────────────────────────────────────────────
✅        WARN     analytics    feature_completeness       All 21 required analytics columns present and non-null


---
## 6 — CLI Usage (reference)

```bash
# Normal monthly run (what the scheduler triggers)
python src/pipeline.py --mode incremental

# Full re-ingest (for quarterly refresh or after schema changes)
python src/pipeline.py --mode full

# Override the incremental start date
python src/pipeline.py --mode incremental --since 2024-10-01

# Check what would run without making changes
python src/pipeline.py --dry-run

# Skip model training (data layer only)
python src/pipeline.py --mode incremental --no-train

# Strict freshness check (fails if data > 2 days old)
python src/pipeline.py --mode incremental --strict-freshness
```

## 7 — GitHub Actions Automation

See `.github/workflows/monthly-pipeline.yml`.

**How it works:**
- Cron schedule: `0 6 1 * *` = 06:00 UTC on the 1st of every month
- Restores previous month's DuckDB from GitHub Artifacts (incremental mode works)
- Runs `python src/pipeline.py --mode incremental`
- Uploads updated state + logs + predictions as new artifacts
- Commits `predictions/YYYY-MM.csv` back to the repo

**Can also be triggered manually** from the repo's Actions tab (`workflow_dispatch`) with optional `mode` and `since` inputs.

---

## ✅ Day 5 Deliverables

| File | Lines | Purpose |
|------|-------|---------|
| `src/pipeline.py` | ~400 | End-to-end orchestrator |
| `src/quality_checks.py` | ~300 | 6 automated quality gates |
| `src/modeling.py` | ~200 | Train + predict (stub for Day 6) |
| `src/database.py` | updated | `create_raw_tables_if_absent()`, `INSERT OR REPLACE` |
| `notebooks/day_05_pipeline.ipynb` | this | E2E run demo + architecture |
| `.github/workflows/monthly-pipeline.yml` | ~80 | Automatic monthly scheduling |

### 📋 Day 6 Handoff

- **The pipeline is done.** Day 6 only touches `src/modeling.py` — replace the `BaselinePredictor` class with real XGBoost / RandomForest. Keep the `.fit(X, y)` / `.predict_proba(X)` interface and everything else continues to work.
- **Target table**: `analytics.monthly_summary` (180 rows × ~25 columns)
- **Target column**: `high_risk_month` (binary, 0/1)
- **Train/test strategy**: stratified k-fold CV with class stratification; reserve 2024 as holdout

---
*End of Day 5 — push, PR, enable the GitHub Actions workflow in your repo settings.*